# SNOOSE
### This notebook queries JPL Horizons for an object's ephemeris over a specified UTC time range and visualizes:
#### - Sky positions (RA/Dec) as magenta cross markers
#### - 3σ positional uncertainties as red ellipses
#### The visualization is rendered using Aladin Lite and embedded directly in the notebook via a base64 HTML IFrame.

### User Inputs
#### Object Identifier ID:
##### - Examples: "3I", "P/2020 MK4", "73P-AB"
##### - For fragmented objects - specify which piece :)
#### Observatory Location:
##### - MPC Observatory Code
##### - Examples: Rubin Observatory = "X05", Apache Point Observatory = "705"
#### Epoch:
##### - In UTC
#####   "start": "YYYY-MM-DD HH:MM",
#####    "stop":  "YYYY-MM-DD HH:MM",
#####    "step":  "1m" | "5m" | "1h" | etc.

##### 3I, asteroid, fragment 
##### write up how this works, what stuff you need to provide, providing 
##### check if uncertainty ellipse works 

In [10]:
from astroquery.jplhorizons import Horizons
from IPython.display import HTML, display
import pandas as pd
import json
import uuid
import re

# User Inputs
TARGET_ID = "79P"
OBS_CODE  = "705"
EPOCHS = {
    "start": "2025-11-13 06:20",  # UTC
    "stop":  "2025-11-13 06:30",  # UTC
    "step":  "1m",
}
QUANTITIES = "1,3,9,36,41"


def resolve_target_id(target_id, location, epochs, quantities=None):
    """
    Try querying Horizons with the provided target_id.
    If the target name is ambiguous (common for comets), parse the candidate
    records from the error message and choose the record whose epoch year is
    closest to the current year.

    Returns
    -------
    resolved_id : str
        The original target_id if no ambiguity, otherwise the selected record #.
    eph : astropy Table
        The Horizons ephemerides table.
    resolution_note : str or None
        Message describing the fallback choice, if used.
    """
    try:
        h = Horizons(id=target_id, location=location, epochs=epochs)
        try:
            eph = h.ephemerides(quantities=quantities)
        except TypeError:
            eph = h.ephemerides()
        return target_id, eph, None

    except ValueError as e:
        msg = str(e)

        if "Ambiguous target name" not in msg:
            raise

        # Parse candidate table lines like:
        # 90000395    2022    29P    29P    Schwassmann-Wachmann 1
        pattern = re.compile(
            r"^\s*(\d+)\s+(\d{4})\s+(.+?)\s+(.+?)\s+(.+?)\s*$",
            re.MULTILINE
        )
        matches = pattern.findall(msg)

        # Fallback to stricter split-based parsing if regex misses anything
        candidates = []
        for m in matches:
            recno = m[0].strip()
            epoch_yr = int(m[1].strip())
            candidates.append({"record": recno, "epoch_yr": epoch_yr})

        if not candidates:
            # Split-based parser for robustness
            for line in msg.splitlines():
                line = line.rstrip()
                if not line or not line.lstrip().startswith(tuple("0123456789")):
                    continue
                parts = line.split()
                if len(parts) >= 2 and parts[0].isdigit() and parts[1].isdigit():
                    candidates.append({
                        "record": parts[0],
                        "epoch_yr": int(parts[1]),
                    })

        if not candidates:
            raise ValueError(
                "Target name is ambiguous, but candidate record numbers could not be parsed.\n"
                f"Original Horizons message:\n{msg}"
            )

        current_year = pd.Timestamp.utcnow().year

        # Choose the epoch year closest to the current year
        best = min(
            candidates,
            key=lambda x: (abs(x["epoch_yr"] - current_year), -x["epoch_yr"])
        )

        resolved_id = best["record"]
        chosen_epoch_yr = best["epoch_yr"]

        h = Horizons(id=resolved_id, location=location, epochs=epochs)
        try:
            eph = h.ephemerides(quantities=quantities)
        except TypeError:
            eph = h.ephemerides()

        resolution_note = (
            f'Target "{target_id}" was ambiguous in Horizons, so it was '
            f'automatically resolved to record #{resolved_id} '
            f'(epoch year {chosen_epoch_yr}, closest to current year {current_year}).'
        )

        return resolved_id, eph, resolution_note


# Resolve target first
RESOLVED_ID, eph, resolution_note = resolve_target_id(
    TARGET_ID, OBS_CODE, EPOCHS, quantities=QUANTITIES
)

raw = eph.to_pandas()

# Pick Anomaly Column
def pick_anomaly_column(columns):
    cols = list(columns)
    lower = {c: c.strip().lower() for c in cols}

    # Mean anomaly preferred
    if "M" in cols:
        return "M", "Mean anomaly"
    for c in cols:
        if "mean" in lower[c] and "anom" in lower[c]:
            return c, "Mean anomaly"

    # True anomaly fallback
    if "nu" in cols:
        return "nu", "True anomaly"
    for c in cols:
        if "true" in lower[c] and "anom" in lower[c]:
            return c, "True anomaly"

    return None, None


anom_col, anom_label = pick_anomaly_column(raw.columns)
if anom_col is None:
    anom_like = [c for c in raw.columns if ("anom" in c.lower()) or (c.strip().lower() in {"m", "nu"})]
    raise KeyError(
        "No mean/true anomaly column found in Horizons output.\n"
        f"Anomaly-like columns present: {anom_like}\n"
        "Debug with:\n"
        "  print(raw.columns.tolist())"
    )

# Pack for JS
df = raw.rename(columns={
    "RA": "RA_deg",
    "DEC": "DEC_deg",
    "RA_3sigma": "RA3_asec",
    "DEC_3sigma": "DEC3_asec",
    anom_col: "anom_deg",
})

required = ["RA_deg", "DEC_deg", "RA3_asec", "DEC3_asec", "datetime_str", "anom_deg"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise KeyError(f"Missing required columns: {missing}\nAvailable columns: {df.columns.tolist()}")

ra0 = df["RA_deg"].mean()
dec0 = df["DEC_deg"].mean()

records = df[["RA_deg", "DEC_deg", "datetime_str", "RA3_asec", "DEC3_asec", "anom_deg"]].to_dict("records")
js_array = json.dumps(records)

resolution_note_js = json.dumps(resolution_note if resolution_note else "")
target_label_js = json.dumps(f"{TARGET_ID} → {RESOLVED_ID}" if RESOLVED_ID != TARGET_ID else TARGET_ID)

uid = uuid.uuid4().hex[:8]
wrap_id = f"wrap_{uid}"
aladin_id = f"aladin_{uid}"
controls_id = f"controls_{uid}"
sidepanel_id = f"sidepanel_{uid}"
rot_id = f"rot_{uid}"
flip_id = f"flip_{uid}"
time_id = f"sp_time_{uid}"
radec_id = f"sp_radec_{uid}"
ra3_id = f"sp_ra3_{uid}"
dec3_id = f"sp_dec3_{uid}"
anom_id = f"sp_anom_{uid}"
status_id = f"status_{uid}"
target_id_span = f"target_{uid}"
note_id = f"note_{uid}"

html = f"""
<div id="{wrap_id}" style="position:relative; width:1100px; height:650px; overflow:hidden; font-family:Arial,sans-serif; background:#f5f5f5; border:0;">
  <div id="{aladin_id}" style="width:100%; height:100%;"></div>

  <div id="{controls_id}" style="
      position:absolute;
      top:10px;
      right:320px;
      background:rgba(255,255,255,0.88);
      padding:8px 10px;
      border:1px solid #ccc;
      z-index:9999;">
    <label>Rotate:
      <input type="range" id="{rot_id}" min="0" max="360" value="0">
    </label><br>
    <label>
      <input type="checkbox" id="{flip_id}"> Flip Horizontally
    </label>
  </div>

  <div id="{sidepanel_id}" style="
      position:absolute;
      top:10px;
      right:10px;
      width:300px;
      background:rgba(255,255,255,0.92);
      border:1px solid #ccc;
      padding:10px 12px;
      box-sizing:border-box;
      z-index:9999;">
    <h3 style="margin:0 0 8px 0; font-size:16px;">Ephemeris Point</h3>
    <div style="margin:6px 0; font-size:13px; line-height:1.25;"><b style="display:inline-block; width:105px;">Target:</b> <span id="{target_id_span}">—</span></div>
    <div style="margin:6px 0; font-size:13px; line-height:1.25;"><b style="display:inline-block; width:105px;">Time:</b> <span id="{time_id}">—</span></div>
    <div style="margin:6px 0; font-size:13px; line-height:1.25;"><b style="display:inline-block; width:105px;">RA, Dec:</b> <span id="{radec_id}">—</span></div>
    <div style="margin:6px 0; font-size:13px; line-height:1.25;"><b style="display:inline-block; width:105px;">3σ RA:</b> <span id="{ra3_id}">—</span></div>
    <div style="margin:6px 0; font-size:13px; line-height:1.25;"><b style="display:inline-block; width:105px;">3σ Dec:</b> <span id="{dec3_id}">—</span></div>
    <div style="margin:6px 0; font-size:13px; line-height:1.25;"><b style="display:inline-block; width:105px;">{anom_label}:</b> <span id="{anom_id}">—</span></div>
    <div id="{note_id}" style="margin-top:8px; font-size:12px; color:#8a5a00; line-height:1.35;"></div>
    <div id="{status_id}" style="margin-top:8px; font-size:12px; color:#666;">
      Starting...
    </div>
  </div>
</div>

<script>
(function() {{
  const data = {js_array};
  const resolutionNote = {resolution_note_js};
  const targetLabel = {target_label_js};

  function setText(id, txt) {{
    const el = document.getElementById(id);
    if (el) el.textContent = txt;
  }}

  function updateStatus(msg) {{
    setText("{status_id}", msg);
  }}

  function updatePanel(d) {{
    setText("{target_id_span}", targetLabel);
    setText("{time_id}", d.datetime_str);
    setText("{radec_id}", d.RA_deg.toFixed(6) + ", " + d.DEC_deg.toFixed(6));
    setText("{ra3_id}", d.RA3_asec.toFixed(2) + " arcsec");
    setText("{dec3_id}", d.DEC3_asec.toFixed(2) + " arcsec");
    setText("{anom_id}", d.anom_deg.toFixed(3) + " deg");
    setText("{note_id}", resolutionNote);
  }}

  window.addEventListener("error", function(e) {{
    updateStatus("JS error: " + e.message);
  }});

  function bootAladin() {{
    if (typeof A === "undefined") {{
      updateStatus("Aladin JS failed to load.");
      return;
    }}

    A.init.then(() => {{
      updateStatus("Loading Aladin viewer...");

      const aladin = A.aladin("#{aladin_id}", {{
        survey: "P/DSS2/color",
        target: "{ra0:.6f} {dec0:.6f}",
        fov: 0.5,
        cooFrame: "ICRS"
      }});

      const trackLayer = A.catalog({{
        name: "Track",
        shape: "cross",
        color: "magenta",
        sourceSize: 10
      }});
      aladin.addCatalog(trackLayer);

      const overlay = A.graphicOverlay({{
        color: "red",
        lineWidth: 2
      }});
      aladin.addOverlay(overlay);

      if (data.length > 0) {{
        updatePanel(data[0]);
      }}

      data.forEach(d => {{
        const m = A.marker(d.RA_deg, d.DEC_deg, {{
          popupTitle: d.datetime_str,
          popupDesc:
            "{anom_label}: " + d.anom_deg.toFixed(3) + "°"
            + "<br>3σ_RA: " + d.RA3_asec.toFixed(2) + "″"
            + "<br>3σ_Dec: " + d.DEC3_asec.toFixed(2) + "″"
        }});
        m.data = d;
        trackLayer.addSources([m]);

        const raSigDeg  = d.RA3_asec / 3600.0;
        const decSigDeg = d.DEC3_asec / 3600.0;
        overlay.add(A.ellipse(d.RA_deg, d.DEC_deg, raSigDeg, decSigDeg, 0, {{}}));
      }});

      try {{
        trackLayer.on("click", (source) => {{
          if (source && source.data) {{
            updatePanel(source.data);
          }}
        }});
      }} catch (e) {{
        console.error("Catalog click handler failed:", e);
      }}

      const rotSlider  = document.getElementById("{rot_id}");
      const flipToggle = document.getElementById("{flip_id}");

      let rotDeg = 0;
      let flipX = false;
      let renderEl = null;

      function findRenderEl() {{
        const root = document.getElementById("{aladin_id}");
        const canvas = root.querySelector("canvas");
        if (!canvas) return null;
        return canvas.parentElement || canvas;
      }}

      function applyTransform() {{
        if (!renderEl) return;
        const sx = flipX ? -1 : 1;
        renderEl.style.transformOrigin = "center center";
        renderEl.style.transform = `scaleX(${{sx}}) rotate(${{rotDeg}}deg)`;
      }}

      function ensureRenderElAndApply() {{
        renderEl = findRenderEl();
        if (renderEl) {{
          applyTransform();
          return true;
        }}
        return false;
      }}

      if (!ensureRenderElAndApply()) {{
        const obs = new MutationObserver(() => {{
          if (ensureRenderElAndApply()) {{
            obs.disconnect();
          }}
        }});
        obs.observe(document.getElementById("{aladin_id}"), {{
          childList: true,
          subtree: true
        }});
      }}

      rotSlider.addEventListener("input", () => {{
        rotDeg = parseFloat(rotSlider.value) || 0;
        applyTransform();
      }});

      flipToggle.addEventListener("change", () => {{
        flipX = !!flipToggle.checked;
        applyTransform();
      }});

      updateStatus("Click a marker to update.");
    }}).catch(err => {{
      console.error(err);
      updateStatus("A.init failed: " + err);
    }});
  }}

  // Load Aladin script once, then boot
  if (typeof A === "undefined") {{
    const s = document.createElement("script");
    s.src = "https://aladin.cds.unistra.fr/AladinLite/api/v3/latest/aladin.js";
    s.onload = bootAladin;
    s.onerror = function() {{
      updateStatus("Could not load Aladin script.");
    }};
    document.head.appendChild(s);
  }} else {{
    bootAladin();
  }}
}})();
</script>
"""

display(HTML(html))